# The WIND Magnetic Field Investigation — Implementation / 구현

**Paper**: Lepping, R. P. et al., *The WIND Magnetic Field Investigation*, Space Sci. Rev. **71**, 207–229, 1995. DOI: 10.1007/BF00751330

This notebook implements three core algorithms from the paper:
1. **Dual triaxial fluxgate noise model** — sensor noise PSD with the paper's measured floor (∼2 × 10⁻⁶ nT²/Hz above 10 Hz) and the algebraic 1/r³ spacecraft-field separation.
2. **ICME / magnetic-cloud signature detection** — Bz smooth-rotation + low fluctuation + strong |B| heuristic following Burlaga (1981, 1991).
3. **GSE → GSM rotation** — exact dipole-tilt rotation about the X axis, used by the MFI ground processing (Fig. 5 K.P. flow).

이 노트북은 논문의 세 가지 핵심 알고리즘을 구현한다: (1) 듀얼 삼축 플럭스게이트 잡음 모델 + 1/r³ 우주선 자기장 분리, (2) Burlaga 기준의 ICME/자기 구름 검출, (3) GSE→GSM 좌표 회전.

**Conda env**: `study-with-ai`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numpy.typing import NDArray
from typing import Tuple

plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11
rng = np.random.default_rng(seed=42)

## Part 1: Dual triaxial fluxgate noise model / 듀얼 삼축 플럭스게이트 잡음 모델

Per Fig. 3b of the paper: above 10 Hz the prototype sensor PSD is essentially **flat at 2 × 10⁻⁶ nT²/Hz**, with total RMS ≈ 12.1 × 10⁻³ nT integrated over 0–50 Hz. Below ∼1 Hz a 1/f component appears; we model it as a piecewise PSD with a knee at ~1 Hz.

Fig. 3b에 따르면 10 Hz 이상에서 잡음 PSD는 약 2 × 10⁻⁶ nT²/Hz 평탄, 0–50 Hz 적분 RMS ≈ 12.1 × 10⁻³ nT. 1 Hz 이하 1/f 성분 존재, 1 Hz 부근 knee로 모델링.

We then synthesize 0–22 Hz magnetometer noise time-series from this PSD and compare with the paper's specifications (<0.006 nT r.m.s. over 0–10 Hz).

In [ ]:
def mfi_noise_psd(freqs: NDArray, psd_floor: float = 2e-6, knee_hz: float = 1.0) -> NDArray:
    """Return MFI prototype-sensor noise PSD in nT^2/Hz.

    Models the noise as 1/f below ``knee_hz`` and flat at ``psd_floor`` above it,
    consistent with Lepping et al. (1995) Fig. 3b.

    Args:
        freqs: Array of frequencies in Hz (must be > 0).
        psd_floor: Flat-region PSD level in nT^2/Hz. Default 2e-6 from paper.
        knee_hz: 1/f knee frequency in Hz.

    Returns:
        PSD array in nT^2/Hz, same shape as ``freqs``.
    """
    psd = np.where(freqs >= knee_hz, psd_floor, psd_floor * (knee_hz / np.maximum(freqs, 1e-3)))
    return psd


def synthesize_noise_timeseries(
    duration_s: float,
    fs_hz: float = 44.0,
    seed: int = 0,
) -> Tuple[NDArray, NDArray]:
    """Generate a 1-axis magnetometer noise time series from the MFI noise model.

    Args:
        duration_s: Time-series duration in seconds.
        fs_hz: Sample rate in Hz; default 44 (MFI internal sample rate).
        seed: RNG seed for reproducibility.

    Returns:
        (t, b) where t is time in s and b is field in nT.
    """
    n = int(duration_s * fs_hz)
    if n % 2 == 1:
        n += 1
    freqs = np.fft.rfftfreq(n, d=1.0 / fs_hz)
    freqs[0] = freqs[1]  # avoid div-by-zero at DC
    psd = mfi_noise_psd(freqs)
    # Convert PSD to amplitude per bin: |X|^2 = PSD * fs * n / 2 (one-sided)
    amp = np.sqrt(psd * fs_hz * n / 2.0)
    local_rng = np.random.default_rng(seed)
    phase = np.exp(1j * 2 * np.pi * local_rng.random(len(freqs)))
    spectrum = amp * phase
    b = np.fft.irfft(spectrum, n=n)
    t = np.arange(n) / fs_hz
    return t, b


# Synthesize 60 s of noise and check the integrated RMS
t, b_noise = synthesize_noise_timeseries(duration_s=60.0, fs_hz=44.0, seed=1)
rms_total = np.std(b_noise)
# Paper: 12.1 mNT total RMS over 0-50 Hz; here we use 0-22 Hz (Nyquist), so expect ~ sqrt(22/50)*12.1 ~ 8 mNT
print(f"Synthesized noise RMS over 0-22 Hz: {rms_total*1e3:.2f} mNT")
print(f"Paper's 0-10 Hz spec:                <6 mNT")

fig, ax = plt.subplots(2, 1, figsize=(10, 7))
ax[0].plot(t[:1000], b_noise[:1000] * 1e3, lw=0.6)
ax[0].set_xlabel("Time [s]")
ax[0].set_ylabel("B noise [mNT]")
ax[0].set_title("MFI synthesized sensor noise time series (1 axis)")
ax[0].grid(alpha=0.3)

freqs_plot = np.logspace(-2, np.log10(22), 200)
ax[1].loglog(freqs_plot, mfi_noise_psd(freqs_plot), lw=2, label="Model PSD")
ax[1].axhline(2e-6, color="k", ls=":", label="Paper Fig. 3b floor 2e-6 nT$^2$/Hz")
ax[1].set_xlabel("Frequency [Hz]")
ax[1].set_ylabel("PSD [nT$^2$/Hz]")
ax[1].set_title("MFI noise PSD model (Lepping et al. 1995, Fig. 3b)")
ax[1].legend()
ax[1].grid(which="both", alpha=0.3)
plt.tight_layout()
plt.show()

### 1.b Dual-magnetometer 1/r³ S/C field separation / 듀얼 자력계 1/r³ 우주선 자기장 분리

Eq. (M2): for a dipolar S/C field the linear combination $\mathbf{B}_{\text{ambient}} = (r_I^3 \mathbf{B}_{IB} - r_O^3 \mathbf{B}_{OB})/(r_I^3 - r_O^3)$ removes the contamination algebraically.

쌍극자 우주선 자기장은 거리^3 에 반비례하므로, OB(12 m), IB(8 m) 측정값의 선형 결합으로 분리 가능.

In [ ]:
def remove_spacecraft_dipole(
    b_outboard: NDArray,
    b_inboard: NDArray,
    r_outboard_m: float = 12.0,
    r_inboard_m: float = 8.0,
) -> NDArray:
    """Algebraically remove a dipolar spacecraft-field contribution from dual-magnetometer data.

    Uses Lepping et al. (1995) Eq. (M2) of the notes:
        B_ambient = (r_I^3 * B_IB - r_O^3 * B_OB) / (r_I^3 - r_O^3)
    valid when the spacecraft contribution at each sensor scales as 1/r^3.

    Args:
        b_outboard: OB-sensor reading (any shape; vector or time series).
        b_inboard:  IB-sensor reading, same shape as ``b_outboard``.
        r_outboard_m: OB sensor radial distance from S/C center (m). Default 12.
        r_inboard_m:  IB sensor radial distance from S/C center (m). Default 8.

    Returns:
        Estimated ambient field, same shape as inputs.
    """
    r_o3 = r_outboard_m**3
    r_i3 = r_inboard_m**3
    return (r_i3 * b_inboard - r_o3 * b_outboard) / (r_i3 - r_o3)


# Worked example mirroring the notes appendix
B_true = np.array([5.0, 0.0, 2.0])  # nT, true ambient
alpha = 0.05  # nT * m^3 effective dipole strength
B_OB = B_true + alpha / 12.0**3 * np.array([0.0, 0.0, 1.0]) * 1728  # alpha/r_O^3 -> nT
B_IB = B_true + alpha / 8.0**3 * np.array([0.0, 0.0, 1.0]) * 512    # alpha/r_I^3 -> nT
# (Note: simplified scalar dipole; demo only)
B_OB = B_true + 0.05 * np.array([0, 0, 1])
B_IB = B_true + 0.05 * (12.0 / 8.0)**3 * np.array([0, 0, 1])

B_ambient_est = remove_spacecraft_dipole(B_OB, B_IB, r_outboard_m=12.0, r_inboard_m=8.0)
print(f"True ambient B   : {B_true}")
print(f"OB reading       : {B_OB}")
print(f"IB reading       : {B_IB}")
print(f"Estimated ambient: {B_ambient_est}")
print(f"Residual error   : {B_ambient_est - B_true} nT")
assert np.allclose(B_ambient_est, B_true, atol=1e-9)

## Part 2: ICME / magnetic-cloud signature detection / 행성간 CME / 자기 구름 신호 검출

Following Burlaga (1981, 1991), a magnetic cloud is identified by three simultaneous criteria:

1. **Strong field**: $|B|$ exceeds a threshold (typically 10 nT at 1 AU).
2. **Smooth rotation**: $B_z$ rotates smoothly through ∼180° over ∼12 hr (low high-frequency variance).
3. **Low fluctuation**: relative field fluctuation $\sigma_B/|B|$ is low (∼<0.1).

Burlaga(1981, 1991) 기준: (1) 강한 자기장 (|B| > 10 nT @ 1 AU), (2) 부드러운 Bz 회전 (≃180°, ∼12 hr), (3) 낮은 변동 (σ_B/|B| < 0.1).

We synthesize a quiet-solar-wind segment with an embedded magnetic-cloud Lundquist-flux-rope passage and run the detector on it.

In [ ]:
def lundquist_flux_rope(
    t_hr: NDArray,
    t_center_hr: float,
    duration_hr: float = 12.0,
    b0_nT: float = 18.0,
    inclination_deg: float = 30.0,
) -> Tuple[NDArray, NDArray, NDArray]:
    """Generate a synthetic magnetic-cloud (Lundquist flux-rope) field profile.

    Args:
        t_hr: Time array in hours.
        t_center_hr: Hour at which the spacecraft crosses the rope axis.
        duration_hr: Total duration of the cloud passage in hours.
        b0_nT: Peak axial field strength in nT.
        inclination_deg: Rotation axis tilt; controls how much rotation appears
            in Bz vs By.

    Returns:
        (Bx, By, Bz) arrays in GSE coordinates, in nT, same shape as ``t_hr``.
    """
    tau = (t_hr - t_center_hr) / (duration_hr / 2.0)
    inside = np.abs(tau) <= 1.0
    # Bessel-J0 axial component, J1 azimuthal — first-zero scaling alpha*R = 2.405
    from scipy.special import j0, j1

    arg = 2.405 * tau
    bz_axis = b0_nT * j0(arg)
    bphi = b0_nT * j1(arg)
    inc = np.deg2rad(inclination_deg)
    Bx = np.where(inside, 0.4 * bphi * np.sin(inc), 0.0)
    By = np.where(inside, bphi * np.cos(inc), 0.0)
    Bz = np.where(inside, bz_axis * np.cos(inc) - bphi * np.sin(inc), 0.0)
    return Bx, By, Bz


def detect_magnetic_cloud(
    t_hr: NDArray,
    Bx: NDArray,
    By: NDArray,
    Bz: NDArray,
    b_threshold_nT: float = 10.0,
    fluctuation_threshold: float = 0.1,
    rotation_min_deg: float = 90.0,
    window_hr: float = 1.0,
) -> NDArray:
    """Flag samples belonging to a magnetic cloud per Burlaga (1981, 1991) criteria.

    A sliding window of ``window_hr`` hours evaluates:
      (1) mean |B| > ``b_threshold_nT``;
      (2) sigma_B / |B|  <= ``fluctuation_threshold``;
      (3) total angular rotation of B-vector across the window
          >= ``rotation_min_deg`` (smooth rotation surrogate).

    Args:
        t_hr: Time array in hours, monotonically increasing.
        Bx, By, Bz: Magnetic-field components (nT) in any orthonormal frame.
        b_threshold_nT: Minimum mean-|B| threshold.
        fluctuation_threshold: Maximum tolerated sigma_B / |B|.
        rotation_min_deg: Minimum cumulative angular rotation across the window.
        window_hr: Sliding-window width in hours.

    Returns:
        Boolean array of cloud-flag, same length as ``t_hr``.
    """
    n = len(t_hr)
    dt_hr = np.median(np.diff(t_hr))
    win_n = max(int(window_hr / dt_hr), 3)
    Bmag = np.sqrt(Bx**2 + By**2 + Bz**2)
    flag = np.zeros(n, dtype=bool)
    for i in range(win_n // 2, n - win_n // 2):
        sl = slice(i - win_n // 2, i + win_n // 2)
        bm_mean = np.mean(Bmag[sl])
        if bm_mean < b_threshold_nT:
            continue
        bm_std = np.std(Bmag[sl])
        if bm_std / max(bm_mean, 1e-3) > fluctuation_threshold:
            continue
        # cumulative rotation angle of (Bx,By,Bz) unit vector
        ux = Bx[sl] / np.maximum(Bmag[sl], 1e-9)
        uy = By[sl] / np.maximum(Bmag[sl], 1e-9)
        uz = Bz[sl] / np.maximum(Bmag[sl], 1e-9)
        cos_steps = ux[1:] * ux[:-1] + uy[1:] * uy[:-1] + uz[1:] * uz[:-1]
        cos_steps = np.clip(cos_steps, -1.0, 1.0)
        total_rot_deg = np.degrees(np.sum(np.arccos(cos_steps)))
        if total_rot_deg >= rotation_min_deg:
            flag[sl] = True
    return flag


# Build synthetic 48-hr time series at 1-min cadence
dt_min = 1.0
t_hr_full = np.arange(0, 48.0, dt_min / 60.0)
# background slow solar wind
Bx_bg = 3.0 + 0.5 * rng.standard_normal(t_hr_full.size)
By_bg = -2.0 + 0.5 * rng.standard_normal(t_hr_full.size)
Bz_bg = 0.0 + 0.5 * rng.standard_normal(t_hr_full.size)

# embedded magnetic cloud centered at hour 24, duration 12 hr
Bx_mc, By_mc, Bz_mc = lundquist_flux_rope(t_hr_full, t_center_hr=24.0, duration_hr=12.0,
                                          b0_nT=18.0, inclination_deg=30.0)

Bx = Bx_bg + Bx_mc
By = By_bg + By_mc
Bz = Bz_bg + Bz_mc
Bmag = np.sqrt(Bx**2 + By**2 + Bz**2)

flag = detect_magnetic_cloud(t_hr_full, Bx, By, Bz,
                              b_threshold_nT=10.0,
                              fluctuation_threshold=0.15,
                              rotation_min_deg=90.0,
                              window_hr=2.0)

fig, ax = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
ax[0].plot(t_hr_full, Bmag, label="|B|", color="k")
ax[0].plot(t_hr_full, Bz, label="$B_z$", color="tab:blue", lw=0.9)
ax[0].axhline(0, color="grey", ls=":")
ax[0].fill_between(t_hr_full, -25, 25, where=flag, color="tab:orange", alpha=0.25, label="MC flag")
ax[0].set_ylabel("B [nT]")
ax[0].set_title("Synthetic 48-hr WIND/MFI-like time series with embedded magnetic cloud")
ax[0].legend(loc="upper right")
ax[0].grid(alpha=0.3)
ax[0].set_ylim(-25, 25)

ax[1].plot(t_hr_full, Bx, label="$B_x$", lw=0.8)
ax[1].plot(t_hr_full, By, label="$B_y$", lw=0.8)
ax[1].plot(t_hr_full, Bz, label="$B_z$", lw=0.8)
ax[1].fill_between(t_hr_full, -25, 25, where=flag, color="tab:orange", alpha=0.25)
ax[1].set_xlabel("Time [hr]")
ax[1].set_ylabel("B components [nT]")
ax[1].legend(loc="upper right")
ax[1].grid(alpha=0.3)
ax[1].set_ylim(-25, 25)
plt.tight_layout()
plt.show()

frac_flagged = flag.mean()
print(f"Fraction of time flagged as magnetic cloud: {frac_flagged*100:.1f}%")
print("Expected ≈ 12 hr / 48 hr = 25.0% — detection should bracket the cloud passage.")

## Part 3: GSE → GSM rotation / GSE → GSM 좌표 회전

MFI ground processing (Fig. 5, paper p. 225) outputs both GSE and GSM. The two systems share the X-axis (Earth-Sun) and differ only by a rotation about X by the dipole-tilt angle μ.

MFI 지상 처리(Fig. 5)는 GSE와 GSM 좌표계 모두 출력. 두 좌표계는 X축(지구-태양)을 공유하며 dipole tilt 각 μ만큼 X축 둘레로 회전.

We implement an exact rotation given μ and verify on canonical test vectors.

In [ ]:
def gse_to_gsm(b_gse: NDArray, mu_deg: float) -> NDArray:
    """Rotate a 3-vector from GSE to GSM coordinates.

    GSM is GSE rotated about the shared X-axis by the dipole-tilt angle ``mu_deg``
    such that the Earth's magnetic-dipole vector lies in the GSM X-Z plane.

    Args:
        b_gse: Field vector in GSE, shape (3,) or (..., 3).
        mu_deg: Dipole tilt angle in degrees, range typically [-35, 35].

    Returns:
        Field vector in GSM, same shape as ``b_gse``.
    """
    mu = np.deg2rad(mu_deg)
    R = np.array([
        [1.0, 0.0, 0.0],
        [0.0, np.cos(mu), np.sin(mu)],
        [0.0, -np.sin(mu), np.cos(mu)],
    ])
    return np.tensordot(b_gse, R.T, axes=([-1], [0]))


def dipole_tilt_angle_deg(day_of_year: int, hour_utc: float) -> float:
    """Approximate IGRF dipole-tilt angle in degrees.

    Simplified analytic form using a 23.4 deg obliquity and 11.5 deg dipole offset.
    Adequate for educational ±2 deg accuracy; production code should call IGRF.

    Args:
        day_of_year: 1..366.
        hour_utc: Decimal hour 0..24.

    Returns:
        Dipole-tilt angle mu in degrees.
    """
    # Solar declination (deg)
    decl = 23.44 * np.sin(np.deg2rad(360.0 * (day_of_year - 80) / 365.25))
    # Diurnal swing of geographic dipole projection
    diurnal = 11.5 * np.cos(np.deg2rad(15.0 * (hour_utc - 5.0)))
    return decl + diurnal


# Test 1: identity at mu=0
b_test = np.array([1.0, 2.0, 3.0])
assert np.allclose(gse_to_gsm(b_test, 0.0), b_test)

# Test 2: pure Z component at mu=90 should rotate to -Y
b_z = np.array([0.0, 0.0, 1.0])
rotated = gse_to_gsm(b_z, 90.0)
print(f"GSE [0,0,1] @ mu=90deg -> GSM {rotated} (expect [0, 1, 0])")

# Test 3: pure Y at mu=90 should rotate to +Z
b_y = np.array([0.0, 1.0, 0.0])
rotated = gse_to_gsm(b_y, 90.0)
print(f"GSE [0,1,0] @ mu=90deg -> GSM {rotated} (expect [0, 0, -1])")

# Demonstrate on the synthetic magnetic cloud above: convert from "GSE" to "GSM" with mu=20deg
mu_deg_constant = 20.0
b_gse = np.column_stack([Bx, By, Bz])
b_gsm = gse_to_gsm(b_gse, mu_deg_constant)

fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
ax[0].plot(t_hr_full, Bz, label="$B_z$ GSE", color="tab:blue")
ax[0].plot(t_hr_full, b_gsm[:, 2], label="$B_z$ GSM (mu=20°)", color="tab:red", ls="--")
ax[0].axhline(0, color="grey", ls=":")
ax[0].set_ylabel("$B_z$ [nT]")
ax[0].set_title("GSE→GSM transformation effect on $B_z$ (relevant for storm forecast)")
ax[0].legend()
ax[0].grid(alpha=0.3)

ax[1].plot(t_hr_full, By, label="$B_y$ GSE", color="tab:blue")
ax[1].plot(t_hr_full, b_gsm[:, 1], label="$B_y$ GSM (mu=20°)", color="tab:red", ls="--")
ax[1].axhline(0, color="grey", ls=":")
ax[1].set_xlabel("Time [hr]")
ax[1].set_ylabel("$B_y$ [nT]")
ax[1].legend()
ax[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Approximate dipole-tilt at DOY=80, 12 UTC: {dipole_tilt_angle_deg(80, 12.0):.2f} deg")

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | Modern Equivalent / 현대 동등물 |
|---|---|---|
| Sensor noise floor / 센서 잡음 한계 | <0.006 nT r.m.s. (0–10 Hz), ∼2×10⁻⁶ nT²/Hz | PSP/FIELDS MAG, Solar Orbiter MAG (∼1×10⁻⁶ nT²/Hz) |
| Dual-magnetometer S/C-field separation / 듀얼 자력계 우주선 자기장 분리 | OB+IB linear combination, 1/r³ | STEREO/MAG, Solar Orbiter MAG dual sensors |
| Dynamic range via range switching / 레인지 절환 동적 범위 | 8 ranges × 12-bit → 156 dB end-to-end | DSCOVR MAG, PSP/FIELDS use similar gain-stage logic |
| Magnetic-cloud detection / 자기 구름 검출 | Burlaga (1981) criteria — strong B + smooth rotation + low fluctuation | NOAA SWPC operational forecast products use the same heuristic |
| Coordinate frames / 좌표계 | GSE+GSM standard products | All ISTP-era and modern heliospheric data continue both products |
| Pre-trigger snapshot memory / 사전-트리거 스냅샷 메모리 | 256 kbit, 82 s pre-trigger | MMS, PSP, Solar Orbiter use circular high-rate buffers |

### Key takeaways from the implementation / 구현으로부터의 핵심 결론
1. The MFI noise floor is so low (∼6 mNT integrated 0–10 Hz) that it is below all known IMF fluctuation levels at 1 AU — the instrument is *not* the limiting factor in IMF studies.
2. The 1/r³ algebraic separation works exactly when the spacecraft contribution is purely dipolar; non-dipolar residuals require statistical methods (Mariani group's drift study).
3. A simple sliding-window detector with three Burlaga criteria correctly brackets the synthetic 12-hr magnetic cloud passage at the right time, illustrating why MFI's KP cadence (1 vec / 92 s) suffices for cloud monitoring.
4. The GSE↔GSM transform changes Bz substantially when |μ| is large; for storm prediction this is the difference between *triggering reconnection* and *missing it*.

구현의 핵심 결론: (1) MFI 잡음 한계는 1 AU의 어떤 IMF 변동보다도 낮아 IMF 연구의 *제한 요인이 아님*. (2) 1/r³ 해석적 분리는 우주선 기여가 *순수 dipolar*일 때만 정확. (3) Burlaga 3-기준 sliding-window 검출기가 12-hr 합성 자기 구름을 정확히 식별. (4) GSE↔GSM 변환은 |μ|가 클 때 Bz를 크게 바꾸어 폭풍 예측에 결정적.